# Does the winner survive a change of map?

Every result this project has — the policy shootout, the window sweep, the L1
ablation, all of it — comes from **one random map**, `map_seed=42`. That is the
largest external-validity gap in the work: a ranking established on a single
map instance is an anecdote about that instance until it is shown to hold
elsewhere.

This notebook runs the same comparison over many maps and asks three
questions:

1. **Does the ranking hold?** Or does the winner change with the map?
2. **Is W900 still the right window**, or was that a property of map 42?
3. **How much does the map matter** relative to the demand seed? If
   between-map variance dominates, then any single-map figure — including
   every number in the docs — carries an error bar nobody has drawn.

It runs the sweep *and* draws the results, because at ten workers the whole
thing is minutes.

## Before you start

- **Prefer a terminal to Jupyter for the run.** Workers are forked, and
  forking a threaded process can deadlock a child; `run.sh` is single-threaded
  and a Jupyter kernel is not.
- Stopping is safe at any point. Every finished trial is already on disk, and
  re-running continues rather than restarting.
- Each map seed is a **different world**, so its runs never merge with
  another map's.

## Configuration

In [ ]:
import time

from src.experiment.store import DEFAULT_RUNS_DIR

# --- The maps ------------------------------------------------------------
# 42 is the map every earlier experiment used; it is first so the sweep also
# reproduces the existing result rather than only extending it.
MAP_SEEDS = [42] + [s for s in range(1, 25) if s != 42]
# stop_density=0.5 is REQUIRED, not decorative: it makes this the 21-stop
# "campus map" every earlier result used. The generator defaults to 0.15,
# which on this seed gives 6 stops -- a map so easy that plain fixed route
# serves 100% and every comparison flattens into noise.
MAP_SHAPE = dict(n_seeds=17, bounds=(0.0, 0.0, 2000.0, 2000.0),
                 stop_density=0.5)

# --- Demand --------------------------------------------------------------
# Fewer seeds than the single-map campaign's 60: precision now comes from
# maps as well as seeds, and 25 x 40 is 1000 trials per policy per regime.
SEEDS = list(range(40))
BASE_TRIAL = dict(
    max_requests=200,
    capacity=40,
    demand_interval_s=60.0,
    max_time=60000.0,
)

# The regime that discriminates. With unlimited patience every policy serves
# 100% and fulfilment says nothing; 900 s spread the field 80-100% on map 42.
REGIMES = {"impatient": 900.0}

RUNS = DEFAULT_RUNS_DIR
N_JOBS = 10

In [ ]:
import math
import statistics
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import pandas as pd
from cycler import cycler
from scipy import stats

from src.algorithm.ConstrainedInsertionPolicy import ConstrainedInsertionPolicy
from src.algorithm.FixedRoutePolicy import FixedRoutePolicy
from src.algorithm.ReorderingInsertionPolicy import ReorderingInsertionPolicy
from src.algorithm.SkipStopFixedRoutePolicy import SkipStopFixedRoutePolicy
from src.algorithm.WindowInsertionPolicy import WindowInsertionPolicy
from src.experiment import run_campaign, summarize_results
from src.experiment.store import load_runs, to_results
from src.map.graph import PlanarStreetGraph

from IPython.display import Markdown, display

# Validated palette slots 1-3; see notebooks/results.ipynb for the check.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
PRIMARY = SERIES[0]
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.prop_cycle": cycler(color=SERIES),
    "axes.edgecolor": INK_SOFT, "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "grid.linewidth": 0.8, "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110,
})

POLICIES = [
    ("Fixed route", lambda g: FixedRoutePolicy(g)),
    ("Skip-stop", lambda g: SkipStopFixedRoutePolicy(g)),
    ("Constrained", lambda g: ConstrainedInsertionPolicy(g)),
    ("Window W750", lambda g: WindowInsertionPolicy(g, max_wait_s=750.0)),
    ("Window W900", lambda g: WindowInsertionPolicy(g, max_wait_s=900.0)),
    ("Window W1200", lambda g: WindowInsertionPolicy(g, max_wait_s=1200.0)),
    ("Reordering W900",
     lambda g: ReorderingInsertionPolicy(g, max_wait_s=900.0)),
]

print(f"{len(MAP_SEEDS)} maps x {len(POLICIES)} policies x {len(SEEDS)} seeds "
      f"x {len(REGIMES)} regime = "
      f"{len(MAP_SEEDS) * len(POLICIES) * len(SEEDS) * len(REGIMES):,} trials")

## Run it

One campaign per map. The map's generation parameters go into the world, so a
run on map 7 can never be confused with the same policy on map 8 — which is
the whole point of the exercise.

In [ ]:
def world_for(map_seed):
    """
    Describe the map this campaign runs on.
    :param map_seed: Seed handed to PlanarStreetGraph.generate.
    :return: The world identity.
    """
    return {
        "map": "planar",
        "n_seeds": MAP_SHAPE["n_seeds"],
        "bounds": list(MAP_SHAPE["bounds"]),
        "map_seed": map_seed,
        "stop_density": MAP_SHAPE["stop_density"],
        "backend": "MockBackend",
    }


started = time.perf_counter()
ran = skipped = 0
for index, map_seed in enumerate(MAP_SEEDS, start=1):
    graph = PlanarStreetGraph.generate(**MAP_SHAPE, seed=map_seed)
    for regime, patience in REGIMES.items():
        result = run_campaign(
            policies=POLICIES, seeds=SEEDS, world=world_for(map_seed),
            directory=RUNS, graph=graph, n_jobs=N_JOBS, progress=False,
            max_wait_s=patience, **BASE_TRIAL,
        )
        ran += result.ran
        skipped += result.skipped
    print(f"[{index:>2}/{len(MAP_SEEDS)}] map {map_seed:<3} "
          f"ran={ran:<6} skipped={skipped:<6} "
          f"{(time.perf_counter() - started) / 60:5.1f} min elapsed")

print(f"\ndone: {ran} run, {skipped} already recorded, "
      f"{(time.perf_counter() - started) / 60:.1f} min")

## Collect what landed

In [ ]:
LABELS = [label for label, _ in POLICIES]
PATIENCE = REGIMES["impatient"]

wanted_maps = set(MAP_SEEDS)
records = [
    r for r in load_runs(RUNS)
    if r.identity.world.get("backend") == "MockBackend"
    and r.identity.world.get("map_seed") in wanted_maps
    and r.identity.trial.get("max_wait_s") == PATIENCE
    and r.identity.trial.get("max_requests") == BASE_TRIAL["max_requests"]
    and r.identity.policy in LABELS
]

by_map = defaultdict(list)
for record in records:
    by_map[record.identity.world["map_seed"]].append(record)

print(f"{len(records)} runs across {len(by_map)} maps")


def mean_of(runs, field):
    """
    Average one KPI over a set of runs.
    :param runs: RunRecords.
    :param field: KPISnapshot attribute name.
    :return: The mean, or NaN when nothing is defined.
    """
    values = [getattr(r.kpi, field) for r in runs
              if getattr(r.kpi, field) is not None]
    return statistics.fmean(values) if values else float("nan")


# One row per (map, policy): the map's verdict on that policy.
rows = []
for map_seed, runs in by_map.items():
    grouped = defaultdict(list)
    for record in runs:
        grouped[record.identity.policy].append(record)
    for label, policy_runs in grouped.items():
        rows.append({
            "map_seed": map_seed,
            "policy": label,
            "served": mean_of(policy_runs, "served_fraction") * 100,
            "wait": mean_of(policy_runs, "mean_wait_time"),
            "ride": mean_of(policy_runs, "mean_ride_time"),
            "throughput": mean_of(policy_runs, "throughput"),
            "trials": len(policy_runs),
        })
per_map = pd.DataFrame(rows)
display(per_map.head())

## 1. Does the ranking hold?

A policy is ranked on each map by **served fraction first, then mean wait** —
coverage is the criterion the design review states, and wait only means
anything among the passengers actually carried.

In [ ]:
def ranked(frame, metric_high="served", metric_low="wait"):
    """
    Rank policies within each map.
    :param frame: One row per (map, policy).
    :param metric_high: Column where higher is better, ranked first.
    :param metric_low: Column used to break ties, lower being better.
    :return: The frame with a `rank` column, 1 = best.
    """
    out = frame.copy()
    out["rank"] = (
        out.groupby("map_seed")
        .apply(lambda g: g.sort_values([metric_high, metric_low],
                                       ascending=[False, True])
               .assign(r=range(1, len(g) + 1))["r"],
               include_groups=False)
        .reset_index(level=0, drop=True)
    )
    return out


ranks = ranked(per_map)
wins = Counter(ranks[ranks["rank"] == 1]["policy"])
summary = pd.DataFrame({
    "median rank": ranks.groupby("policy")["rank"].median(),
    "best rank": ranks.groupby("policy")["rank"].min(),
    "worst rank": ranks.groupby("policy")["rank"].max(),
    "maps won": pd.Series(wins).reindex(
        ranks["policy"].unique()).fillna(0).astype(int),
}).sort_values(["median rank", "maps won"], ascending=[True, False])
display(summary)

In [ ]:
import numpy as np

# Every map's rank, drawn as a point, with the median marked. A box plot
# alone hides the single most important case: a policy ranked identically on
# every map has zero spread, so its box has zero width and renders as nothing
# at all -- perfect stability, drawn as blank space.
order = list(summary.index)
rng = np.random.default_rng(0)
figure, axis = plt.subplots(figsize=(9, 0.52 * len(order) + 1.6))

for row, label in enumerate(order):
    values = ranks[ranks["policy"] == label]["rank"].to_numpy()
    jitter = rng.uniform(-0.16, 0.16, size=len(values))
    axis.scatter(values, np.full(len(values), row) + jitter, s=26,
                 color=PRIMARY, alpha=0.45, edgecolors="none", zorder=2)
    median = float(np.median(values))
    axis.scatter([median], [row], marker="|", s=520, linewidths=2.4,
                 color=INK, zorder=3)
    spread = values.max() - values.min()
    axis.text(len(order) + 0.55, row,
              "same rank everywhere" if spread == 0 else
              f"ranks {values.min()}-{values.max()}",
              va="center", fontsize=8, color=INK_SOFT)

axis.set_yticks(range(len(order)), order)
axis.invert_yaxis()
axis.set_xticks(range(1, len(order) + 1))
axis.set_xlim(0.5, len(order) + 2.6)
axis.set_xlabel("rank within a map (1 = best; each dot is one map)")
axis.set_title(f"Rank stability across {len(by_map)} maps")
axis.grid(axis="x")
axis.set_axisbelow(True)
axis.spines["left"].set_visible(False)
axis.tick_params(axis="y", length=0)
figure.tight_layout()
plt.show()

**How to read it.** A policy whose box is a single line at one rank is stable:
the map does not change its standing. A wide box means the winner depends on
which map you happened to draw — and if that is true of the leaders, then no
single-map result in this project should be quoted without it.

## 2. Is W900 still the right window?

On map 42 coverage peaked at W900 and fell off both sides. If that optimum
moves from map to map, then the window is not a constant to be set once but a
parameter to be tuned per deployment — a materially different conclusion.

In [ ]:
windows = {"Window W750": 750, "Window W900": 900, "Window W1200": 1200}
window_rows = per_map[per_map["policy"].isin(windows)].copy()
window_rows["W"] = window_rows["policy"].map(windows)

best = (window_rows.sort_values(["served", "wait"], ascending=[False, True])
        .groupby("map_seed").first()["W"])
counts = best.value_counts().sort_index()

figure, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].bar([str(w) for w in counts.index], counts.values, color=PRIMARY,
            width=0.55)
axes[0].set_title("Which window wins, by map")
axes[0].set_xlabel("service window (s)")
axes[0].set_ylabel("maps")
axes[0].grid(axis="y")
axes[0].set_axisbelow(True)
for x, v in enumerate(counts.values):
    axes[0].text(x, v + 0.1, str(v), ha="center", fontsize=8, color=INK_SOFT)

for offset, (label, colour) in enumerate(zip(windows, SERIES)):
    subset = window_rows[window_rows["policy"] == label].sort_values(
        "map_seed")
    axes[1].plot(subset["map_seed"], subset["served"], marker="o",
                 markersize=4, lw=1.6, color=colour, label=label)
axes[1].set_title("Served fraction per map")
axes[1].set_xlabel("map seed")
axes[1].set_ylabel("%")
axes[1].grid(axis="y")
axes[1].set_axisbelow(True)
axes[1].legend()
figure.tight_layout()
plt.show()

display(Markdown(
    f"**{counts.idxmax()} s wins on {counts.max()} of {len(best)} maps.**"))

## 3. How much does the map matter?

Every figure in the docs is a single-map figure. This asks what that is worth:
the spread *between* maps against the spread *within* one, for the leader.

If between-map variance dominates, a single-map result carries an error bar
nobody has drawn — and the honest way to quote any KPI becomes "across maps",
not "on map 42".

In [ ]:
LEADER = summary.index[0]
leader_runs = [r for r in records if r.identity.policy == LEADER]

within = []
for map_seed, runs in by_map.items():
    values = [r.kpi.mean_wait_time for r in runs
              if r.identity.policy == LEADER
              and r.kpi.mean_wait_time is not None]
    if len(values) > 1:
        within.append(statistics.stdev(values))
between = statistics.stdev(
    per_map[per_map["policy"] == LEADER]["wait"].dropna())
within_typical = statistics.fmean(within)

display(Markdown(f"""
For **{LEADER}**, mean wait time:

| | s |
|---|---|
| spread between maps (sd of per-map means) | **{between:.1f}** |
| spread within a map (typical sd across seeds) | {within_typical:.1f} |
| ratio | **{between / within_typical:.2f}x** |
"""))

figure, axis = plt.subplots(figsize=(9, 3.4))
ordered = per_map[per_map["policy"] == LEADER].sort_values("wait")
axis.bar(range(len(ordered)), ordered["wait"], color=PRIMARY, width=0.7)
axis.axhline(ordered["wait"].mean(), color=INK_SOFT, lw=1.2, ls="--")
axis.set_xticks(range(len(ordered)),
                [str(int(s)) for s in ordered["map_seed"]], rotation=90)
axis.set_xlabel("map seed (sorted)")
axis.set_ylabel("mean wait (s)")
axis.set_title(f"{LEADER}: the same policy on {len(ordered)} maps")
axis.grid(axis="y")
axis.set_axisbelow(True)
figure.tight_layout()
plt.show()

## 4. Pooled across every map

The headline table, now resting on many maps rather than one. Confidence
intervals are taken over all trials pooled, so they include between-map
variation — which is the honest interval for "this policy, on a map like
these" rather than "on map 42".

In [ ]:
pooled = to_results(records, paired=False)
display(Markdown(summarize_results(pooled).markdown))

print(f"{'policy':<20}{'served':>9}{'wait':>10}{'ride':>10}"
      f"{'wait CI':>10}")
for label in summary.index:
    runs = pooled[label]
    waits = [r.kpi.mean_wait_time for r in runs
             if r.kpi.mean_wait_time is not None]
    half = (stats.t.ppf(0.975, len(waits) - 1) * statistics.stdev(waits)
            / len(waits) ** 0.5)
    print(f"{label:<20}{mean_of(runs, 'served_fraction'):>8.1%}"
          f"{mean_of(runs, 'mean_wait_time'):>10.0f}"
          f"{mean_of(runs, 'mean_ride_time'):>10.0f}"
          f"{100 * half / statistics.fmean(waits):>9.2f}%")

## 5. Does the success criterion hold on every map?

§3.2.2 asks for fulfilment at or above 80% and a positive sum of percentage
differences against a fixed-route bus. Passing on one map is an anecdote;
passing on all of them is a result.

In [ ]:
BASELINE = "Fixed route"
verdicts = []
for map_seed, runs in sorted(by_map.items()):
    grouped = defaultdict(list)
    for record in runs:
        grouped[record.identity.policy].append(record)
    if BASELINE not in grouped:
        continue
    base = grouped[BASELINE]
    for label, policy_runs in grouped.items():
        if label == BASELINE:
            continue
        wait = ((mean_of(base, "mean_wait_time")
                 - mean_of(policy_runs, "mean_wait_time"))
                / mean_of(base, "mean_wait_time"))
        ride = ((mean_of(base, "mean_ride_time")
                 - mean_of(policy_runs, "mean_ride_time"))
                / mean_of(base, "mean_ride_time"))
        thru = ((mean_of(policy_runs, "throughput")
                 - mean_of(base, "throughput"))
                / mean_of(base, "throughput"))
        fulfil = mean_of(policy_runs, "fulfillment_rate") * 100
        verdicts.append({
            "map_seed": map_seed, "policy": label, "fulfilment": fulfil,
            "sum %": (wait + ride + thru) * 100,
            "pass": fulfil >= 80 and (wait + ride + thru) > 0,
        })

criteria = pd.DataFrame(verdicts)
display(criteria.groupby("policy").agg(
    maps=("pass", "size"),
    passed=("pass", "sum"),
    worst_fulfilment=("fulfilment", "min"),
    worst_sum=("sum %", "min"),
).sort_values("passed", ascending=False))

## What to conclude

Fill this in once it has run. The three things worth writing down:

- whether the leader is the same policy on every map, or only on most;
- whether the winning window is constant, or drifts;
- whether any figure quoted elsewhere in the project needs a between-map
  error bar attached to it.

If the ranking is stable, every single-map result in the docs is safe to
quote and this notebook is the citation for saying so. If it is not, that is
a more important finding than any individual KPI.